# HIPAA Security Rule Changes RAG Assistant

This RAG application enables healthcare IT professionals to query in plain English the 2025 HIPAA Security Rule NPRM against the existing rule, receiving cited answers that identify what is changing and how to implement those changes.



In [ ]:
# Install dependencies (be patient--can take up to 90 seconds)
!pip install pymupdf sentence-transformers chromadb google-genai -q

print('Dependencies installed')

# Load necessary packages
import fitz #PyMuPDF - PDF parsing
from sentence_transformers import SentenceTransformer # Embedding model
import chromadb # Vector store
import google.genai as genai # Gemini API
import os # File path management
from google.colab import userdata # Secure API key access
import requests # PDF download from URLs
print("All imports successful")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:
# Mount Google Drive for storage
from google.colab import drive
drive.mount('/content/drive')

print('Drive mounted')

Mounted at /content/drive
Drive mounted and ChromaDB initialized


In [ ]:
# Initialize persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path='/content/drive/MyDrive/hipaa_rag/chroma_db'
)

print('ChromaDB initialized')

In [ ]:
# Define documents and URLs
DOCS = {
    'nprm_2025.pdf": "https://www.govinfo.gov/content/pkg/FR-2025-01-06/pdf/2024-30983.pdf',
    'hipaa_security_rule_current.pdf": "https://www.hhs.gov/sites/default/files/hipaa-simplification-201303.pdf'
}

# Create directory for storing document PDFs
PDF_DIR = '/content/drive/MyDrive/hipaa_rag/docs'
os.makedirs(PDF_DIR, exist_ok=True)


# Download PDFs from URLs
for filename, url in DOCS.items():
    filepath = os.path.join(PDF_DIR, filename)
    if os.path.exists(filepath):
      print(f'{filename} already exists -- skipping.')
    else:
      print(f'Downloading {filename}...')
      r = requests.get(url)
      if r.status_code != 200:
        print(f'Failed to download {filename}. Status code: {r.status_code}. Skipping ingestion for this file.')
        continue
      with open(filepath, 'wb') as f:
        f.write(r.content)
      print(f'Done.')


nprm_2025.pdf already exists -- skipping.
hipaa_security_rule_current.pdf already exists -- skipping.


In [ ]:

# Initialize embedding model from HuggingFace
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Embedding model initialized')



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized


In [ ]:
# Initialize Gemini client using the API key from Colab secrets
GOOGLE_API_KEY=userdata.get('Gemini') # Replace with name of your API key
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print('Gemini client initialized.')

Gemini client initialized with API Key


In [ ]:
# Initialize ChromaDB collection
collection = chroma_client.get_or_create_collection(
    name='hipaa_docs',
    metadata={'hnsw:space': 'cosine'}
)

print('ChromaDB collection initialized')

Models and collection initialized


In [ ]:
def ingest_document(pdf_path, source_label):
  """
  Load a PDF, chunk it, embed chunks, and store in ChromaDB.
  source_label identifies which document chunks came from.
  """
  # Open PDF and extract text page by page
  doc = fitz.open(pdf_path)

  # Create empty lists
  chunks = []
  metadatas = []
  ids = []

  # Loop through individual pages of each doc, chunk by page
  for page_num, page in enumerate(doc):
    text=page.get_text() # Extract text from each page

    # Skip pages with very little text (headers, footers, blank pages)
    if len(text.strip()) < 100:
      continue

    chunks.append(text) # Append chunks to list
    metadatas.append({ # Create metadata for each chunk
        'source': source_label, # Source document for chunk
        'page': page_num + 1 # Page number of chunk within source document
    })
    ids.append(f"{source_label}_page_{page_num + 1}") # IDs to be used for citations in query response

  # Embed all chunks
  print(f'Embedding {len(chunks)} pages from {source_label}...')
  embeddings = embedding_model.encode(chunks).tolist() # ChromaDB expects list of vectors

  # Store in ChromaDB
  collection.add(
      documents=chunks,
      embeddings=embeddings,
      metadatas=metadatas,
      ids=ids
  )

  print(f'Done. {len(chunks)} chunks stored for {source_label}')

In [ ]:
def retrieve_chunks(query, k=5):
  """
  Embed the query and retrieve the k most similar chunks from ChromaDB.
  Returns raw ChromaDB results including documents and metadata.
  """
  # Embed the query using same model as ingestion
  embeddings = embedding_model.encode(query) # Embed the query

  # Query ChromaDB for k nearest neighbors by cosin similarity
  results = collection.query(
    query_embeddings=[embeddings.tolist()],
    n_results=k
)
  # Return results to be passed to generate_response
  return results


In [ ]:
def generate_response(query, results):
  """
  Build contgext from retrieved chunks and generated a grounded, cited response using Gemini.

  Args:
    query (str): The user's plain-English question
    results(dict): ChromaDB results from retrieve_chunks()

  Returns:
    str: A cited response grounded in the provided regulatory text.
  """

  # Define system prompt for response
  system_prompt = """
    -You are a helpful healthcare privacy assistant with legal knowledge.
    -You are helping someone with knowledge of healthcare privacy and healthcare IT, but not strong legal knowledge.
    -You are helping an organization that is compliant with current HIPAA regulations but is unsure of how the proposed changes will mean for them.

    Core Rules:
    -Answer questions using only the provided context and cite your sources in responses.
    -Never provide incomplete answers or hallucinate answers.
    -Do not provide answers that are not supported by the provided context.
    -Where appropriate, identify specific changes from the current rule to the proposed new rule.
    -Only answer questions that can be answered with the context.
    -If the question cannot be answered with the context, respond "I am not allowed to answer that question."
    -Your response may include directions to implement the specific proposed changes.
    -All responses with directions must help accomplish a specific goal in the proposed changes.

    Output Rules:
    -Responses should include a brief summary (1-2 sentences) responding to the question.
    -Responses should identify actionable steps for implementing proposed changes.
    -If the query does not require actionable steps, end your response with "No actions are required."
    """
  # Extract documents and metadata from reults
  documents = results['documents'][0]
  metadata_list = results['metadatas'][0]
  # Parse documents and metadata_list results into context to be used to respond to query
  context = ""
  for document, metadata in zip(documents, metadata_list):
      context += f'[Source: {metadata['source']}, Page: {metadata['page']}]\n{document}\n\n'

  # Pass context and query to Gemini and return cited response
  response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    config=genai.types.GenerateContentConfig(
        system_instruction=system_prompt
    ),
    contents=f'Context:\n{context}\n\nQuestion: {query}' # Format message to LLM to define context and query
  )

  return response.text # Return text of LLM response

In [ ]:
# Enter your query here
query = 'What are three most significant changes with the new rule?'
results = retrieve_chunks(query)
answer = generate_response(query, results)
print(answer)